# NB04 — Gate safety & cascade cost *(scientific core, CPU)*

A triage gate is only clinically acceptable if, when **discarding exams as NORMAL**, it almost does not
hide disease. Here we answer:

1. **Safety × efficiency:** at each threshold, what is the fraction of filtered exams (workload removed) and
   what is the **pathology miss rate** among the discarded ones?
2. **Miss by pathology:** which of the 7 pathologies would be missed, and how many (absolute counts —
   ECTASIA and NEOPLASIA have very few cases).
3. **Net benefit (decision curve):** does the gate beat "refer all" / "refer none"?
4. **2-stage cascade:** operating the gate at target-sensitivity, how much workload does it remove **before**
   the expert reader, and at what safety cost.
5. **By center (LOCAL):** is safety maintained in both centers?

> **Two notions of "error" — we report both:**
> - *Target false-negative:* missing an `ALTERED` exam (binary label).
> - *Pathology missed (severe):* missing an exam with ≥1 of the 7 pathologies. **This is the safety metric.**


In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'configs').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from configs import config as C
from src import metrics as M, plots, utils
import numpy as np, pandas as pd

print('monitored pathologies:', C.PATHOLOGIES)
print('gate target:', C.TARGET, '| ambiguous policy:', C.AMBIGUOUS_POLICY)

## 1. Pool predictions with pathology metadata
Each image appears in exactly 1 test fold → the *pool* covers the dataset 1×. We merge by
`image_name` with the labels of the 7 pathologies, `ALTERED` and `LOCAL`.

In [ ]:
# (a) metadata per image_name (test pool, with drop_neither same as training)
meta_parts = []
for f in range(C.N_FOLDS):
    d = pd.read_csv(C.SPLITS_DIR / f'fold_{f}_test.csv')
    meta_parts.append(d)
meta = pd.concat(meta_parts).drop_duplicates(C.IMG_COL).reset_index(drop=True)
neither = (meta['NORMAL'] == 0) & (meta['ALTERADO'] == 0)
meta = meta[~neither].reset_index(drop=True)
meta['n_patho'] = meta[C.PATHOLOGIES].sum(axis=1)
meta['has_patho'] = (meta['n_patho'] > 0).astype(int)
meta = meta.set_index(C.IMG_COL)
print('images (pool):', len(meta), '| with pathology:', int(meta['has_patho'].sum()))

# (b) pool predictions for a model -> aligned to meta by image_name
def pooled_frame(model_key):
    ys, ps, names, folds = [], [], [], []
    for f in range(C.N_FOLDS):
        z = np.load(C.PRED_DIR / f'{model_key}_fold{f}_preds.npz', allow_pickle=True)
        ys.append(z['y']); ps.append(z['p']); names.append(z['image_name'].astype(str))
        folds.append(np.full(len(z['y']), f))
    df = pd.DataFrame({'image_name': np.concatenate(names),
                       'y': np.concatenate(ys), 'p': np.concatenate(ps),
                       'fold_id': np.concatenate(folds)}).set_index('image_name')
    df = df.join(meta[['ALTERADO', 'has_patho', 'n_patho', 'LOCAL'] + C.PATHOLOGIES], how='left')
    assert df['has_patho'].isna().sum() == 0, 'image missing metadata!'
    return df

# recommended model = best AUPRC from NB03
best = utils.load_json(C.METRICS_DIR / 'consolidated.json')['best_model_auprc']
print('analyzed triage model:', best)
DF = pooled_frame(best)
DF.head(3)

## 2. Safety × efficiency curve (sweeping the threshold)
For each threshold `thr`, the gate discards as NORMAL those with `p < thr`. We measure:
- **filtered fraction** = workload removed;
- **pathology miss-rate** = missed pathologies / total pathologies;
- **target miss-rate** = missed ALTERED / total ALTERED.

In [ ]:
def safety_curve(df, thresholds=None):
    p = df['p'].to_numpy()
    has_patho = df['has_patho'].to_numpy() > 0
    is_alt = df['ALTERADO'].to_numpy() > 0
    n = len(df); tot_patho = has_patho.sum(); tot_alt = is_alt.sum()
    if thresholds is None:
        thresholds = np.linspace(0.0, 1.0, 101)
    rows = []
    for thr in thresholds:
        pred_normal = p < thr            # discarded by the gate
        filt = int(pred_normal.sum())
        miss_patho = int((pred_normal & has_patho).sum())
        miss_alt = int((pred_normal & is_alt).sum())
        rows.append({'threshold': float(thr),
                     'fraction_filtered': filt / n, 'n_filtered': filt,
                     'pathology_missed': miss_patho,
                     'miss_rate_pathology': miss_patho / max(tot_patho, 1),
                     'target_missed': miss_alt,
                     'miss_rate_target': miss_alt / max(tot_alt, 1)})
    return pd.DataFrame(rows)

sc = safety_curve(DF)
# reference points: highest filtered fraction with pathology miss <= 1%, 2%, 5%
for lim in [0.00, 0.01, 0.02, 0.05]:
    ok = sc[sc['miss_rate_pathology'] <= lim]
    if len(ok):
        r = ok.loc[ok['fraction_filtered'].idxmax()]
        print(f'pathology miss <= {lim*100:>4.0f}%  ->  filters {r.fraction_filtered*100:5.1f}% '
              f'of exams (thr={r.threshold:.3f}, {int(r.pathology_missed)} missed pathologies)')
    else:
        print(f'pathology miss <= {lim*100:>4.0f}%  ->  unattainable')

## 3. Figure: safety × efficiency (EN/PT)

In [ ]:
# x-axis = filtered fraction; y-axis = pathology miss-rate (ordered by filtered fraction)
scs = sc.sort_values('fraction_filtered')
plots.plot_gate_safety(scs['fraction_filtered'].to_numpy(),
                       scs['miss_rate_pathology'].to_numpy(),
                       name=f'nb04_safety_efficiency_{best}')
print('figure saved: nb04_safety_efficiency_' + best)

## 4. Operation at target sensitivity: what the gate delivers
At the clinical operating point (target-sens on the **binary target**), we report the filtered workload, the
missed pathology (safety), and the NPV — the metric that matters for "safe discard".

In [ ]:
def get_cv_pred_normal(df, target_sens):
    y = df['y'].to_numpy(); p = df['p'].to_numpy(); fold_ids = df['fold_id'].to_numpy()
    pred_normal = np.zeros_like(p, dtype=bool)
    thrs = []
    for f in range(C.N_FOLDS):
        mask_train = (fold_ids != f)
        mask_test  = (fold_ids == f)
        thr = M.threshold_for_sensitivity(y[mask_train], p[mask_train], target_sens)
        pred_normal[mask_test] = (p[mask_test] < thr)
        thrs.append(thr)
    return pred_normal, np.mean(thrs)

def operate_at_target_sens(df, target_sens):
    pred_normal, mean_thr = get_cv_pred_normal(df, target_sens)
    y = df['y'].to_numpy()
    has_patho = df['has_patho'].to_numpy() > 0
    is_alt = df['ALTERADO'].to_numpy() > 0
    n = len(df)
    
    y_hat_pool = (~pred_normal).astype(int)
    at = M.at_threshold(y, y_hat_pool, 0.5)
    
    return {'target_sens': target_sens, 'threshold': mean_thr,
            'sensitivity': at['sensitivity'], 'specificity': at['specificity'],
            'npv': at['npv'], 'ppv': at['ppv'],
            'fraction_filtered': float(pred_normal.sum() / n),
            'n_filtered': int(pred_normal.sum()),
            'pathology_missed': int((pred_normal & has_patho).sum()),
            'total_pathology': int(has_patho.sum()),
            'miss_rate_pathology': float((pred_normal & has_patho).sum() / max(has_patho.sum(), 1)),
            'target_missed': int((pred_normal & is_alt).sum())}

op_rows = [operate_at_target_sens(DF, s) for s in C.TARGET_SENS]
op_tbl = pd.DataFrame(op_rows)
display(op_tbl.round(3))

## 5. Miss by individual pathology (absolute counts)
At 95% target-sens, **which** pathologies would be missed? Absolute counts because ECTASIA and
NEOPLASIA have very few cases — an isolated % rate would be misleading.

In [ ]:
def per_pathology_miss(df, target_sens):
    pred_normal, mean_thr = get_cv_pred_normal(df, target_sens)
    rows = []
    for path in C.PATHOLOGIES:
        present = df[path].to_numpy() > 0
        total = int(present.sum())
        missed = int((pred_normal & present).sum())
        rows.append({'pathology': path, 'total': total, 'missed': missed,
                     'miss_rate': missed / total if total else np.nan,
                     'detected': total - missed})
    return pd.DataFrame(rows), float(mean_thr)

pp95, thr95 = per_pathology_miss(DF, 0.95)
print(f'At target sensitivity 95% (thr={thr95:.3f}):')
display(pp95.round(3))

## 6. Net benefit (decision curve) — EN/PT

In [ ]:
nb = M.net_benefit(DF['y'].to_numpy(), DF['p'].to_numpy())
plots.plot_net_benefit(nb, name=f'nb04_decision_curve_{best}')
# threshold range where the gate beats 'refer all'
import numpy as np
better = [d for d in nb if d['nb_model'] > d['nb_all'] and d['nb_model'] > 0]
if better:
    print(f'gate beats "refer all" at pt ~[{better[0]["pt"]:.2f}, {better[-1]["pt"]:.2f}]')
print('figure saved: nb04_decision_curve_' + best)

## 7. 2-stage cascade (gate → expert)
In real triage, the gate runs **before** the reader: it discards the predicted NORMALs and refers the rest.
At target-sens, we compute the **removed workload** (exams the expert doesn't need to see) and the **cost**
(missed pathologies). This translates the gate into operational impact.

In [ ]:
def cascade(df, target_sens):
    r = operate_at_target_sens(df, target_sens)
    n = len(df)
    workload_reduction = r['fraction_filtered']          # fraction the gate resolves alone
    referred = 1 - workload_reduction                    # fraction referred to the expert
    return {'target_sens': target_sens,
            'workload_reduction_pct': 100 * workload_reduction,
            'referred_pct': 100 * referred,
            'pathology_missed': r['pathology_missed'],
            'miss_rate_pathology_pct': 100 * r['miss_rate_pathology'],
            'npv': r['npv']}

casc = pd.DataFrame([cascade(DF, s) for s in C.TARGET_SENS])
display(casc.round(2))
print('Reading: at sens 95%, the gate saves ~{:.0f}% of the workload, missing {} pathology(s).'.format(
      casc[casc.target_sens==0.95]['workload_reduction_pct'].iloc[0],
      int(casc[casc.target_sens==0.95]['pathology_missed'].iloc[0])))

## 8. Stratification by center (LOCAL)
Safety needs to hold across both centers (1 and 2), which have different prevalences.

In [ ]:
def by_center(df, target_sens):
    pred_normal, mean_thr = get_cv_pred_normal(df, target_sens)
    rows = []
    for loc, g in df.groupby('LOCAL'):
        loc_mask = (df['LOCAL'] == loc).to_numpy()
        pn_loc = pred_normal[loc_mask]
        has_patho_loc = (df['has_patho'].to_numpy() > 0)[loc_mask]
        is_alt_loc = (df['ALTERADO'].to_numpy() > 0)[loc_mask]
        n = loc_mask.sum()
        
        rows.append({'LOCAL': int(loc), 'n': int(n), 'prevalence_alt': float(is_alt_loc.mean()),
                     'fraction_filtered': float(pn_loc.sum()/n),
                     'pathology_missed': int((pn_loc & has_patho_loc).sum()),
                     'total_pathology': int(has_patho_loc.sum()),
                     'miss_rate_pathology': float((pn_loc & has_patho_loc).sum()/max(has_patho_loc.sum(),1)),
                     'sensitivity': float((~pn_loc & is_alt_loc).sum()/max(is_alt_loc.sum(),1))})
    return pd.DataFrame(rows), float(mean_thr)

cen, thr_cen = by_center(DF, 0.95)
print(f'global threshold (pooled target-sens 95%) = {thr_cen:.3f}, applied to each center:')
display(cen.round(3))

## 9. Consolidate safety analysis in JSON

In [ ]:
safety_out = {
    'model': best,
    'safety_curve': sc.to_dict(orient='records'),
    'operation_at_target_sens': op_rows,
    'per_pathology_miss_at_sens95': {'threshold': thr95,
                                     'rows': pp95.to_dict(orient='records')},
    'net_benefit': nb,
    'cascade': casc.to_dict(orient='records'),
    'by_center_at_sens95': {'threshold': thr_cen, 'rows': cen.to_dict(orient='records')},
    'definitions': {
        'target_missed': 'false-negative of the ALTERED label',
        'pathology_missed': 'exam with >=1 of the 7 pathologies discarded as NORMAL (safety)',
    },
}
utils.save_json(safety_out, C.METRICS_DIR / 'nb04_gate_safety.json')
sc.to_csv(C.METRICS_DIR / 'nb04_safety_curve.csv', index=False)
casc.to_csv(C.METRICS_DIR / 'nb04_cascade.csv', index=False)
print('saved: results/metrics/nb04_gate_safety.json (+ CSVs)')
print('Next: NB05 — synthesis, LaTeX tables and ready numbers for the article.')